In [2]:
import h5py
from pathlib import Path

PROJECT_DIR = Path(
    r"C:\Users\adaly\OneDrive\Documents\DMDeficientGalaxyTNG-1"
)

DATA_DIR = (
    PROJECT_DIR
    / "data"
    / "TNG300-1"
    / "output"
)


offset_files = list(DATA_DIR.glob("*offset*099*.hdf5"))

print("Offset files found:")
for file in offset_files:
    print("  ", file)


if len(offset_files) == 0:
    raise FileNotFoundError(
        "Could not find an offsets_099 HDF5 file."
    )


offset_file = offset_files[0]

print()
print("=" * 70)
print("INSPECTING")
print("=" * 70)
print(offset_file)


with h5py.File(offset_file, "r") as f:

    print()
    print("=" * 70)
    print("HDF5 CONTENTS")
    print("=" * 70)

    def print_structure(name, obj):

        if isinstance(obj, h5py.Dataset):

            print(
                f"DATASET: {name}"
                f" | shape={obj.shape}"
                f" | dtype={obj.dtype}"
            )

        elif isinstance(obj, h5py.Group):

            print(f"GROUP:   {name}")

    f.visititems(print_structure)

Offset files found:
   C:\Users\adaly\OneDrive\Documents\DMDeficientGalaxyTNG-1\data\TNG300-1\output\offsets_099.hdf5

INSPECTING
C:\Users\adaly\OneDrive\Documents\DMDeficientGalaxyTNG-1\data\TNG300-1\output\offsets_099.hdf5

HDF5 CONTENTS
GROUP:   FileOffsets
DATASET: FileOffsets/Group | shape=(600,) | dtype=int64
DATASET: FileOffsets/SnapByType | shape=(600, 6) | dtype=int64
DATASET: FileOffsets/SubLink | shape=(125,) | dtype=int64
DATASET: FileOffsets/SubLink_gal | shape=(116,) | dtype=int64
DATASET: FileOffsets/Subhalo | shape=(600,) | dtype=int64
GROUP:   Group
DATASET: Group/SnapByType | shape=(17625892, 6) | dtype=int64
GROUP:   Subhalo
GROUP:   Subhalo/LHaloTree
DATASET: Subhalo/LHaloTree/File | shape=(14485709,) | dtype=int32
DATASET: Subhalo/LHaloTree/Index | shape=(14485709,) | dtype=int32
DATASET: Subhalo/LHaloTree/Num | shape=(14485709,) | dtype=int32
DATASET: Subhalo/SnapByType | shape=(14485709, 6) | dtype=int64
GROUP:   Subhalo/SubLink
DATASET: Subhalo/SubLink/LastProge

In [3]:
import pandas as pd
PROJECT_DIR = Path(
    r"C:\Users\adaly\OneDrive\Documents\DMDeficientGalaxyTNG-1"
)

OFFSET_FILE = (
    PROJECT_DIR
    / "data"
    / "TNG300-1"
    / "output"
    / "offsets_099.hdf5"
)

REPRESENTATIVE_FILE = (
    PROJECT_DIR
    / "data"
    / "representative_dmdgs.csv"
)

df = pd.read_csv(REPRESENTATIVE_FILE)

print(
    f"Loaded {len(df)} representative galaxies."
)

print()

particle_types = {
    0: "gas",
    1: "DM",
    4: "stars"
}

with h5py.File(OFFSET_FILE, "r") as f:

    subhalo_offsets = f[
        "Subhalo/SnapByType"
    ][()]

    file_offsets = f[
        "FileOffsets/SnapByType"
    ][()]

print("=" * 80)
print("PARTICLE OFFSETS")
print("=" * 80)

results = []


for _, galaxy in df.iterrows():

    subhalo_id = int(galaxy["SubhaloID"])

    print()
    print("-" * 80)
    print(f"Subhalo {subhalo_id}")
    print("-" * 80)

    print(
        f"M_star       = "
        f"{galaxy['M_star_Msun']:.4e} Msun"
    )

    print(
        f"M_total_2Rh  = "
        f"{galaxy['M_total_2Rh']:.6e}"
    )

    print(
        f"M_DM_2Rh     = "
        f"{galaxy['M_DM_2Rh']:.6e}"
    )

    print(
        f"f_DM         = "
        f"{galaxy['f_DM']:.6f}"
    )

    print(
        f"R_half_star  = "
        f"{galaxy['R_half_star']:.4f}"
    )

    print()

    for part_type, name in particle_types.items():
        global_offset = int(
            subhalo_offsets[subhalo_id, part_type]
        )
        length_column = {
            0: "N_gas",
            1: "N_DM",
            4: "N_star"
        }[part_type]

        length = int(
            galaxy[length_column]
        )
        chunk_starts = file_offsets[:, part_type]

        chunk = (
            (chunk_starts <= global_offset)
            .nonzero()[0]
        )[-1]

        local_offset = (
            global_offset
            - chunk_starts[chunk]
        )

        print(
            f"{name:5s}: "
            f"global offset = {global_offset:,} | "
            f"length = {length:,} | "
            f"snapshot chunk = {chunk} | "
            f"local offset = {local_offset:,}"
        )

        results.append({
            "SubhaloID": subhalo_id,
            "ParticleType": name,
            "ParticleTypeNumber": part_type,
            "GlobalOffset": global_offset,
            "Length": length,
            "SnapshotChunk": int(chunk),
            "LocalOffset": int(local_offset)
        })
results_df = pd.DataFrame(results)

output_file = (
    PROJECT_DIR
    / "data"
    / "representative_particle_offsets.csv"
)

results_df.to_csv(
    output_file,
    index=False
)

print()
print("=" * 80)
print("SAVED")
print("=" * 80)
print(output_file)

Loaded 5 representative galaxies.

PARTICLE OFFSETS

--------------------------------------------------------------------------------
Subhalo 738558
--------------------------------------------------------------------------------
M_star       = 2.5894e+09 Msun
M_total_2Rh  = 2.721529e-01
M_DM_2Rh     = 1.035691e-01
f_DM         = 0.380555
R_half_star  = 0.6847

gas  : global offset = 1,503,947,329 | length = 0 | snapshot chunk = 62 | local offset = 17,632,800
DM   : global offset = 2,194,973,182 | length = 28 | snapshot chunk = 84 | local offset = 6,825,164
stars: global offset = 232,514,782 | length = 332 | snapshot chunk = 186 | local offset = 1,035,463

--------------------------------------------------------------------------------
Subhalo 355308
--------------------------------------------------------------------------------
M_star       = 2.4921e+09 Msun
M_total_2Rh  = 2.446458e-01
M_DM_2Rh     = 9.161884e-02
f_DM         = 0.374496
R_half_star  = 1.5633

gas  : global offset = 8

In [5]:
OUTPUT_DIR = (
    PROJECT_DIR
    / "data"
    / "TNG300-1"
    / "output"
    / "groups_099"
)


print("=" * 70)
print("SEARCHING FOR SNAPSHOT 099 FILES")
print("=" * 70)

snapshot_files = sorted(
    OUTPUT_DIR.glob("snap_099.*.hdf5")
)

print(
    f"Found {len(snapshot_files)} snapshot files."
)

print()

for file in snapshot_files:
    print(file)

SEARCHING FOR SNAPSHOT 099 FILES
Found 0 snapshot files.

